# Bronze Layer: Raw Data Ingestion

This notebook ingests raw e-commerce data into Delta Lake bronze tables.

**Tasks:**
- Read CSV files from raw storage
- Apply basic schema enforcement
- Write to bronze Delta tables
- Log ingestion metrics

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

# Initialize Spark session with Delta Lake
spark = SparkSession.builder \
    .appName("BronzeIngestion") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

## Ingest Customers

Load raw customer data into bronze table

In [ ]:
# Define schema
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("customer_segment", StringType(), True)
])

# Read CSV
customers_df = spark.read \
    .option("header", "true") \
    .schema(customer_schema) \
    .csv("/mnt/data/raw/customers.csv")

# Write to Delta bronze table
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/bronze/customers")

## Ingest Products

Load raw product data into bronze table

In [ ]:
# Similar ingestion for products
products_df = spark.read \
    .option("header", "true") \
    .csv("/mnt/data/raw/products.csv")

products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/bronze/products")

## Ingest Orders

Load raw order data into bronze table

In [ ]:
# Ingest orders
orders_df = spark.read \
    .option("header", "true") \
    .csv("/mnt/data/raw/orders.csv")

orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/bronze/orders")

## Ingest Order Items

Load raw order items data into bronze table

In [ ]:
# Define schema for order items
order_item_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("line_total", DoubleType(), True)
])

# Read order items CSV
order_items_df = spark.read \
    .option("header", "true") \
    .schema(order_item_schema) \
    .csv("/mnt/data/raw/order_items.csv")

# Write to Delta bronze table
order_items_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/mnt/delta/bronze/order_items")

## Verification

Check row counts and schema

In [ ]:
print("Bronze tables created:")
print(f"Customers: {spark.read.format('delta').load('/mnt/delta/bronze/customers').count()} rows")
print(f"Products: {spark.read.format('delta').load('/mnt/delta/bronze/products').count()} rows")
print(f"Orders: {spark.read.format('delta').load('/mnt/delta/bronze/orders').count()} rows")
print(f"Order Items: {spark.read.format('delta').load('/mnt/delta/bronze/order_items').count()} rows")